# Day 2: Building a Real-World Telecom RAG System

## 1. The Business Problem: SLA Breaches & High AHT
Telecom customer service agents are required to follow complex Service Level Agreements (SLAs). When a customer complains about an internet outage, the agent must check a 500-page internal manual to determine if they are authorized to dispatch a field technician and offer compensation.

Searching this manual manually takes **5 minutes**, increasing Average Handling Time (AHT) and costing the company millions.

<img src="../RAG_SLA_Business_Impact.png" alt="Business Impact" width="900"/>

**Our Goal:** Build a Retrieval-Augmented Generation (RAG) system that instantly reads the manual and answers the agent's question based on exact internal policies.

## 2. Environment Setup & Data Ingestion
Install and import necessary libraries, load the internal knowledge base text file (`../data/Telecom_Internal_KB.txt`), and split it into chunks.

In [1]:
# Install required libraries
%pip install -q -U \
    langchain \
    langchain-community \
    langchain-core \
    langchain-google-genai \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu \
    python-dotenv \
    tqdm

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ncnn 1.0.20250916 requires portalocker, which is not installed.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

# Multilingual embeddings (Crucial for matching Arabic queries to English text)
print("Loading local embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Load the Knowledge Base
print("Loading Knowledge Base...")
loader = TextLoader('../data/Telecom_Internal_KB.txt', encoding='utf-8')
documents = loader.load()
print(f"✅ Successfully loaded {len(documents)} document(s).")

C:\Users\ToP NeT\AppData\Local\Temp\ipykernel_992\1930259945.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\ToP NeT\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading local embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5731.30it/s]


Loading Knowledge Base...
✅ Successfully loaded 1 document(s).


In [3]:
# Split the text into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(documents)

print(f"✅ Loaded {len(documents)} document.")
print(f"✅ Split into {len(chunks)} chunks.")
print(f"🔍 Sample chunk: \n{chunks[3].page_content}")

✅ Loaded 1 document.
✅ Split into 481 chunks.
🔍 Sample chunk: 
## 2. Hardware Specifications & Router Guides


## 3. Embeddings & Vector Database
Convert the chunks into vector embeddings and store them in a local Vector Store (e.g., ChromaDB or FAISS).

In [4]:
from langchain_community.vectorstores import FAISS
from tqdm import tqdm
import time

print(f"Starting ingestion of {len(chunks)} chunks into FAISS...")

# We ingest in batches to lower resource costs
batch_size = 50
vectorstore = None

for i in tqdm(range(0, len(chunks), batch_size), desc="Embedding & Indexing Chunks"):
    batch = chunks[i:i + batch_size]
    
    if vectorstore is None:
        # First batch initializes the FAISS index
        vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        # Subsequent batches are added to the existing index
        vectorstore.add_documents(batch)
        

# Save the FAISS index locally so we don't have to pay/wait to re-embed later
vectorstore.save_local("faiss_telecom_index")
print("\n✅ Ingestion Complete. FAISS index saved locally.")

# Test if the multilingual retrieval actually works!
print("\nTesting semantic search (Arabic Query -> English Document)...")
test_query = "العميل بيشتكي إن لمبة الراوتر بتنور وتطفي بقالها ٣ أيام"
print(f"\n {test_query}")

results = vectorstore.similarity_search_with_score(test_query, k=10)
print("\n✅ Top match retrieved by FAISS:")
print("--------------------------------------------------")

# Unpack the tuple for the first result
best_doc, best_score = results[0]
print(f"Match Score (Lower distance is better): {best_score:.4f}")
print(best_doc.page_content)
print("--------------------------------------------------")
# Unpack the second result
second_doc, second_score = results[1]
print(f"Match Score: {second_score:.4f}")
print(second_doc.page_content)
print("--------------------------------------------------")



# Unpack the second result
second_doc, second_score = results[2]
print(f"Match Score: {second_score:.4f}")
print(second_doc.page_content)
print("--------------------------------------------------")



# Unpack the second result
second_doc, second_score = results[3]
print(f"Match Score: {second_score:.4f}")
print(second_doc.page_content)
print("--------------------------------------------------")

Starting ingestion of 481 chunks into FAISS...


Embedding & Indexing Chunks: 100%|██████████| 10/10 [00:19<00:00,  1.90s/it]


✅ Ingestion Complete. FAISS index saved locally.

Testing semantic search (Arabic Query -> English Document)...

 العميل بيشتكي إن لمبة الراوتر بتنور وتطفي بقالها ٣ أيام

✅ Top match retrieved by FAISS:
--------------------------------------------------
Match Score (Lower distance is better): 12.1444
## 1. General Service Level Agreement (SLA) & Dispatch Policies
If a customer reports an internet outage (DSL blinking or no sync):
- The L1 agent must first ensure the customer has restarted the router and checked internal wiring.
- If the issue persists for more than 24 hours, the L1 agent must escalate to the Central Exchange team.
- A Field Technician must be dispatched if the line noise margin is below 6dB or if the DSL light is completely off/blinking for 3 consecutive days.
--------------------------------------------------
Match Score: 12.6735
## 4. Cross-Department Escalation Matrix
- **Billing Issues:** Transfer to 111.
- **Fiber Optic Cuts:** Escalate immediately to Tier 3 Fibe

## 4. Prompt Engineering & The RAG Chain
Write the System Prompt enforcing the LLM to act as a telecom support assistant. Inject the retrieved context and the customer's Egyptian Arabic ticket.

In [5]:
%pip install -U langchain-core langchain-google-genai langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import os
from dotenv import load_dotenv

print("Current folder:", os.getcwd())
print(".env exists:", os.path.exists(".env"))

load_dotenv()

key = os.getenv("GOOGLE_API_KEY")
print("API key found:", key is not None)
print("Key length:", len(key) if key else 0)

Current folder: c:\Users\ToP NeT\Desktop\RAG_Demo_Notebook-master\notebooks
.env exists: False
API key found: False
Key length: 0


In [15]:
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

print("Building the Prompt and Gemini RAG Chain...")

# Load variables from the .env file
load_dotenv()

# Set Gemini API Key
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# Initialize the Gemini LLM
gemini_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# The System Prompt (Instructions + Context Injection)
template = """
أنت موظف خدمة عملاء في مزود خدمة إنترنت (ISP). 
مهمتك هي الرد على شكوى العميل باللغة العامية المصرية بطريقة مهذبة واحترافية.
تحذير هام: إياك أن تذكر أي اسم شركة اتصالات حقيقي (مثل اتصالات، فودافون، وي، إلخ) في ردك. قدم نفسك فقط كموظف خدمة عملاء فقط.
يجب عليك استخدام المعلومات الموجودة في (السياق الداخلي) فقط لحل المشكلة.
إذا كانت المشكلة تستدعي إرسال فني حسب القواعد، أخبر العميل بذلك بناءً على السياق.
السياق الداخلي (قوانين الشركة وخطوات الحل):
{context}
شكوى العميل:
{question}
الرد:
"""

prompt = PromptTemplate.from_template(template)
print(f"{prompt.template}")

# Convert FAISS vectorstore into a retriever (pulling top 20 chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

# Helper function to combine the retrieved chunks into one text block
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | gemini_llm | StrOutputParser()
)
print("✅ Gemini RAG Chain is ready!")


Building the Prompt and Gemini RAG Chain...

أنت موظف خدمة عملاء في مزود خدمة إنترنت (ISP). 
مهمتك هي الرد على شكوى العميل باللغة العامية المصرية بطريقة مهذبة واحترافية.
تحذير هام: إياك أن تذكر أي اسم شركة اتصالات حقيقي (مثل اتصالات، فودافون، وي، إلخ) في ردك. قدم نفسك فقط كموظف خدمة عملاء فقط.
يجب عليك استخدام المعلومات الموجودة في (السياق الداخلي) فقط لحل المشكلة.
إذا كانت المشكلة تستدعي إرسال فني حسب القواعد، أخبر العميل بذلك بناءً على السياق.
السياق الداخلي (قوانين الشركة وخطوات الحل):
{context}
شكوى العميل:
{question}
الرد:

✅ Gemini RAG Chain is ready!


## 5. Live Test: Solving the Egyptian Support Ticket
Test the pipeline with a realistic customer complaint.

In [16]:
print("Processing the ticket through Gemini...\n")

customer_ticket = """
أنا دافع الفاتورة من يومين أونلاين والفلوس اتخصمت من الفيزا، 
لكن النت لسه مرجعش لحد دلوقتي ومكتوبلي إن الخدمة موقوفة!
"""

print("Agent AI Response (Gemini):")
print("--------------------------------------------------")

# This sends the ticket to the retriever, formats the prompt, and gets the answer from Gemini
response = rag_chain.invoke(customer_ticket)

print(response)
print("--------------------------------------------------")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Processing the ticket through Gemini...

Agent AI Response (Gemini):
--------------------------------------------------
أهلاً بحضرتك، أنا من خدمة عملاء مزود خدمة الإنترنت.

أنا متفهم جداً للموقف اللي حضرتك فيه وإن النت مش شغال بالرغم من دفع الفاتورة والفلوس اتخصمت من الفيزا.

بالنسبة لمشكلة دفع الفاتورة وتوقف الخدمة، دي بتحتاج تحويل لقسم الفواتير عشان يتأكدوا من وصول المبلغ وتفعيل الخدمة لحضرتك في أسرع وقت.

بعد إذنك، هحول حضرتك على طول لقسم الفواتير عشان يراجعوا الموضوع ويحلوا المشكلة دي. بعتذر جداً عن أي إزعاج حصل لحضرتك.
--------------------------------------------------


In [17]:
print("Processing the ticket through Gemini...\n")

customer_ticket = """
النت شغال بس بطيء جداً وبيظهرلي رسالة على الشاشة فيها كود الخطأ E-204.
أعمل إيه عشان أحل المشكلة دي؟
"""

print("Agent AI Response (Gemini):")
print("--------------------------------------------------")

# This sends the ticket to the retriever, formats the prompt, and gets the answer from Gemini
response = rag_chain.invoke(customer_ticket)

print(response)
print("--------------------------------------------------")

Processing the ticket through Gemini...

Agent AI Response (Gemini):
--------------------------------------------------
أهلاً بيك يا فندم، أنا من خدمة عملاء مزود خدمة الإنترنت.

متفهم جداً المشكلة اللي بتواجهها بخصوص بطء الإنترنت وظهور كود الخطأ E-204.

بالنسبة لكود الخطأ ده، الحل بيكون إن حضرتك تغير إعدادات الـ DNS عندك لـ **8.8.8.8**.

ياريت تجرب الخطوة دي وتبلغني بالنتيجة. لو احتجت أي مساعدة في طريقة التغيير، أنا تحت أمرك وهساعدك خطوة بخطوة.

إحنا موجودين لأي استفسار أو مساعدة تانية.
--------------------------------------------------


# 5. Task 1: Prompt Engineering Experiments

In this experiment, we compare different prompt engineering strategies
for handling the same Egyptian telecom customer support ticket.

The goal is to determine which prompt produces the most accurate,
professional, and policy-compliant response.

In [19]:
customer_ticket = """
العميل بيشتكي إن لمبة الراوتر بتنور وتطفي بقالها ٣ أيام.
أعمل إيه؟
"""

print(customer_ticket)


العميل بيشتكي إن لمبة الراوتر بتنور وتطفي بقالها ٣ أيام.
أعمل إيه؟



### 1: very basic prompt

In [20]:
basic_template = """
Answer the question using the context.

Context:
{context}

Customer Question:
{question}

Answer:
"""

basic_prompt = PromptTemplate.from_template(basic_template)

basic_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | basic_prompt
    | gemini_llm
    | StrOutputParser()
)

basic_response = basic_chain.invoke(customer_ticket)

print("Basic Prompt Response:")
print("-" * 60)
print(basic_response)

Basic Prompt Response:
------------------------------------------------------------
يجب إرسال فني ميداني (Dispatch Field Technician).


### 2: role based prompt

In [21]:
role_template = """
You are a professional telecom customer support agent.

Answer the customer's question using the provided internal context.

Context:
{context}

Customer Question:
{question}

Provide a helpful and professional answer.
"""

role_prompt = PromptTemplate.from_template(role_template)

role_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | role_prompt
    | gemini_llm
    | StrOutputParser()
)

role_response = role_chain.invoke(customer_ticket)

print("Role-Based Prompt Response:")
print("-" * 60)
print(role_response)

Role-Based Prompt Response:
------------------------------------------------------------
أهلاً بك،

بناءً على سياستنا، بما أن لمبة الـ DSL (أو الإنترنت) في الراوتر تومض بشكل مستمر لمدة 3 أيام متتالية، يجب علينا **إرسال فني ميداني** لمعاينة المشكلة وحلها.

قبل ذلك، يرجى التأكد من أن العميل قد قام بالخطوات الأولية التالية:
1.  **إعادة تشغيل الراوتر:** فصل الطاقة عن الراوتر لمدة دقيقة ثم إعادة توصيلها.
2.  **فحص التوصيلات الداخلية:** التأكد من أن جميع الكابلات موصلة بشكل صحيح ومحكم.

إذا استمرت المشكلة بعد هذه الخطوات، فقم بجدولة زيارة فني ميداني.


### 3: Strict RAG Prompt

In [22]:
strict_template = """
أنت موظف خدمة عملاء محترف في مزود خدمة إنترنت.

مهمتك الرد على العميل باللغة العامية المصرية بطريقة مهذبة واحترافية.

القواعد:
1. استخدم المعلومات الموجودة في السياق الداخلي فقط.
2. لا تخترع أي معلومات غير موجودة في السياق.
3. لا تذكر أسماء شركات اتصالات حقيقية.
4. إذا كانت المشكلة تستدعي إرسال فني حسب السياسة، أخبر العميل بذلك.
5. إذا لم يحتوي السياق على إجابة كافية، قل إن المعلومات المتاحة لا تكفي لتحديد الحل.

السياق الداخلي:
{context}

شكوى العميل:
{question}

الرد:
"""

strict_prompt = PromptTemplate.from_template(strict_template)

strict_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | strict_prompt
    | gemini_llm
    | StrOutputParser()
)

strict_response = strict_chain.invoke(customer_ticket)

print("Strict RAG Prompt Response:")
print("-" * 60)
print(strict_response)

Strict RAG Prompt Response:
------------------------------------------------------------
أهلاً بحضرتك يا فندم،

متفهم جداً إن لمبة الراوتر بتنور وتطفي بقالها 3 أيام، وده طبعاً بيأثر على خدمة الإنترنت عند حضرتك.

وفقاً للسياسة المتبعة عندنا، لما لمبة الـ DSL بتفضل تنور وتطفي لمدة 3 أيام متواصلة، ده بيستدعي إننا نبعت لحضرتك فني متخصص عشان يفحص الخط ويحل المشكلة من جذورها.

ممكن بس نأكد على بيانات حضرتك عشان ننسق ميعاد زيارة الفني في أقرب وقت يناسبك؟


In [23]:
print("=" * 80)
print("PROMPT ENGINEERING RESULTS")
print("=" * 80)

print("\n1. BASIC PROMPT")
print(basic_response)

print("\n2. ROLE-BASED PROMPT")
print(role_response)

print("\n3. STRICT RAG PROMPT")
print(strict_response)

PROMPT ENGINEERING RESULTS

1. BASIC PROMPT
يجب إرسال فني ميداني (Dispatch Field Technician).

2. ROLE-BASED PROMPT
أهلاً بك،

بناءً على سياستنا، بما أن لمبة الـ DSL (أو الإنترنت) في الراوتر تومض بشكل مستمر لمدة 3 أيام متتالية، يجب علينا **إرسال فني ميداني** لمعاينة المشكلة وحلها.

قبل ذلك، يرجى التأكد من أن العميل قد قام بالخطوات الأولية التالية:
1.  **إعادة تشغيل الراوتر:** فصل الطاقة عن الراوتر لمدة دقيقة ثم إعادة توصيلها.
2.  **فحص التوصيلات الداخلية:** التأكد من أن جميع الكابلات موصلة بشكل صحيح ومحكم.

إذا استمرت المشكلة بعد هذه الخطوات، فقم بجدولة زيارة فني ميداني.

3. STRICT RAG PROMPT
أهلاً بحضرتك يا فندم،

متفهم جداً إن لمبة الراوتر بتنور وتطفي بقالها 3 أيام، وده طبعاً بيأثر على خدمة الإنترنت عند حضرتك.

وفقاً للسياسة المتبعة عندنا، لما لمبة الـ DSL بتفضل تنور وتطفي لمدة 3 أيام متواصلة، ده بيستدعي إننا نبعت لحضرتك فني متخصص عشان يفحص الخط ويحل المشكلة من جذورها.

ممكن بس نأكد على بيانات حضرتك عشان ننسق ميعاد زيارة الفني في أقرب وقت يناسبك؟


# 6. Task 2: Chunking Strategy Experiments

In this experiment, we compare different chunk sizes and overlap values
to evaluate their effect on semantic retrieval.

The same knowledge base and the same test queries are used for all strategies.


### strategy B: 1000 Character Chunks

In [24]:
splitter_b = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks_b = splitter_b.split_documents(documents)

print("Strategy B")
print("Number of chunks:", len(chunks_b))

Strategy B
Number of chunks: 244


In [25]:
vectorstore_b = FAISS.from_documents(
    chunks_b,
    embeddings
)

print("✅ Strategy B FAISS index created.")

✅ Strategy B FAISS index created.


In [26]:
test_query = "العميل بيشتكي إن لمبة الراوتر بتنور وتطفي بقالها ٣ أيام"

results_a = vectorstore.similarity_search_with_score(
    test_query,
    k=3
)

results_b = vectorstore_b.similarity_search_with_score(
    test_query,
    k=3
)

In [27]:
print("=" * 80)
print("STRATEGY B - 1000/200")
print("=" * 80)

for i, (doc, score) in enumerate(results_b, 1):
    print(f"\nResult {i} | Score: {score:.4f}")
    print(doc.page_content[:500])

STRATEGY B - 1000/200

Result 1 | Score: 12.9484
# Telecom Egypt Internal Technical Support Knowledge Base (Confidential)

## 1. General Service Level Agreement (SLA) & Dispatch Policies
If a customer reports an internet outage (DSL blinking or no sync):
- The L1 agent must first ensure the customer has restarted the router and checked internal wiring.
- If the issue persists for more than 24 hours, the L1 agent must escalate to the Central Exchange team.
- A Field Technician must be dispatched if the line noise margin is below 6dB or if the D

Result 2 | Score: 14.7731
### Router Model: VDF-TP--2020X163
- **Manufacturer:** TP-Link
- **Max Supported Speed:** 100 Mbps
- **DSL Light Behavior:** If blinking, indicates failure to sync with the central exchange. If solid green, sync is successful. If red, physical line cut.
- **Internet Light Behavior:** If red, authentication failure (wrong PPPoE username/password). If off, no IP assigned.
- **Troubleshooting Step 1:** Restart router and w